## Decision tree from scratch

In [2]:
import numpy as np

def entropy(y):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-9))

In [3]:
def information_gain(X_column, y, threshold):
    parent_entropy = entropy(y)

    left_mask = X_column < threshold
    right_mask = X_column >= threshold

    if len(y[left_mask]) == 0 or len(y[right_mask]) == 0:
        return 0

    n = len(y)
    n_left = len(y[left_mask])
    n_right = len(y[right_mask])

    child_entropy = (n_left/n)*entropy(y[left_mask]) + \
                    (n_right/n)*entropy(y[right_mask])

    return parent_entropy - child_entropy

In [4]:
def best_split(X, y):
    best_gain = -1
    split_idx = None
    split_threshold = None

    n_features = X.shape[1]

    for feature_idx in range(n_features):
        X_column = X[:, feature_idx]
        thresholds = np.unique(X_column)

        for threshold in thresholds:
            gain = information_gain(X_column, y, threshold)

            if gain > best_gain:
                best_gain = gain
                split_idx = feature_idx
                split_threshold = threshold

    return split_idx, split_threshold

In [5]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def build_tree(X, y, depth=0, max_depth=3):
    if len(np.unique(y)) == 1 or depth >= max_depth:
        leaf_value = np.bincount(y).argmax()
        return Node(value=leaf_value)

    feature, threshold = best_split(X, y)

    if feature is None:
        leaf_value = np.bincount(y).argmax()
        return Node(value=leaf_value)

    left_mask = X[:, feature] < threshold
    right_mask = X[:, feature] >= threshold

    left_child = build_tree(X[left_mask], y[left_mask], depth+1, max_depth)
    right_child = build_tree(X[right_mask], y[right_mask], depth+1, max_depth)

    return Node(feature, threshold, left_child, right_child)

In [6]:
def predict_sample(node, x):
    if node.value is not None:
        return node.value

    if x[node.feature] < node.threshold:
        return predict_sample(node.left, x)
    else:
        return predict_sample(node.right, x)

def predict(tree, X):
    return np.array([predict_sample(tree, x) for x in X])

In [7]:
from sklearn import datasets
from sklearn.model_selection import train_test_split

iris = datasets.load_iris()
X = iris.data
y = iris.target

# Make binary problem for simplicity
X = X[y != 2]
y = y[y != 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

tree = build_tree(X_train, y_train, max_depth=3)

y_pred = predict(tree, X_test)

accuracy = np.mean(y_pred == y_test)
print("Accuracy:", accuracy)

Accuracy: 1.0
